# 🎙️ Lumini AI Studio - VoxCPM2 Free GPU Server (Kaggle)
### 🚀 100% Free Zero-Shot Studio Voice Cloning Engine (OpenBMB 48kHz)

---
### 📌 Kaggle Free GPU အသုံးပြုနည်း လမ်းညွှန်:
1. ညာဘက် Sidebar မှ **Settings** ➡️ **Accelerator** ➡️ **GPU T4 x2** (သို့) **GPU P100** ရွေးပါ။
2. **Internet** ကို **ON** ဖွင့်ထားကြောင်း စစ်ဆေးပါ (Phone Verification လုပ်ထားရပါမည်)။
3. အောက်ပါ Cell များကို အပေါ်မှအောက်သို့ **Run** နှိပ်ပါ။
4. အောက်ဆုံးတွင် ထွက်လာသော `https://xxxx.ngrok-free.app` URL ကို Lumini Voiceover Studio ထဲသို့ ကူးယူထည့်သွင်းပါ။

### ⚡ Kaggle vs Colab:
| Feature | Kaggle | Colab Free |
|---------|--------|------------|
| GPU Time | **30 hrs/week** | ~4-5 hrs/day |
| GPU Type | T4 x2 or P100 | T4 |
| Session Limit | **12 hrs** | ~90 min idle |
| Internet | ✅ (Verified) | ✅ |

In [ ]:
# ================================================================
# Step 1: GPU စစ်ဆေးခြင်း
# ================================================================
import os, sys, subprocess

# GPU Check
gpu_info = subprocess.getoutput("nvidia-smi -L")
if "NVIDIA" not in gpu_info and "GPU" not in gpu_info:
    print("❌ GPU မတွေ့ရှိပါ! Settings -> Accelerator -> GPU T4 x2 ကို ရွေးချယ်ပြီးမှ ပြန် Run ပါ။")
    sys.exit(1)
else:
    print(f"✅ GPU တွေ့ရှိပါသည်: {gpu_info}")

# Kaggle has a /kaggle/working directory
os.makedirs("/kaggle/working/model_cache", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/working/model_cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/model_cache"
os.environ["TORCH_HOME"] = "/kaggle/working/model_cache"
print("✅ Model cache directory set up!")

In [ ]:
# ================================================================
# Step 2: Packages နှင့် ngrok Tunnel သွင်းယူခြင်း
# ================================================================
print("⏳ လိုအပ်သော AI Packages များ ထည့်သွင်းနေပါသည် (~60s)...", flush=True)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!pip install -q --upgrade pip
!pip install -q voxcpm fastapi uvicorn soundfile torch torchaudio python-multipart pyngrok

print("✅ Packages အားလုံး ထည့်သွင်းပြီးစီးပါပြီ!")

In [ ]:
# ================================================================
# Step 3: ngrok Auth Token ထည့်သွင်းခြင်း
# ================================================================
# 👉 https://dashboard.ngrok.com/get-started/your-authtoken မှ
#    Free Account ဖွင့်ပြီး Auth Token ကူးယူပါ
#
# 💡 Kaggle Secrets သုံးနည်း (ပိုလုံခြုံ):
#    1. ညာဘက် Sidebar -> Add-ons -> Secrets
#    2. Label: NGROK_AUTH_TOKEN, Value: သင်၏ token
#    3. "Attach to this notebook" ကို ON ဖွင့်ပါ

from pyngrok import ngrok

# Method 1: Kaggle Secrets (recommended)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    NGROK_TOKEN = secrets.get_secret("NGROK_AUTH_TOKEN")
    print("✅ ngrok token loaded from Kaggle Secrets!")
except Exception:
    # Method 2: Direct paste (less secure but works)
    NGROK_TOKEN = ""  # <-- သင်၏ ngrok auth token ကို ဒီမှာ paste လုပ်ပါ
    if not NGROK_TOKEN:
        print("⚠️ ngrok Auth Token မထည့်ရသေးပါ!")
        print("👉 https://dashboard.ngrok.com/get-started/your-authtoken")
        print("   မှ Token ကူးယူပြီး NGROK_TOKEN variable ထဲတွင် paste လုပ်ပါ")
        print("   သို့မဟုတ် Kaggle Secrets တွင် NGROK_AUTH_TOKEN အဖြစ် ထည့်သွင်းပါ")
        sys.exit(1)

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok authenticated successfully!")

In [ ]:
# ================================================================
# Step 4: FastAPI Speech Server တည်ဆောက်ခြင်း
# ================================================================
server_script = r'''import os
import io
import gc
import sys
import time
import uuid
import tempfile
from pathlib import Path
from typing import Optional
import soundfile as sf
import torch
import numpy as np
import re
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="Lumini VoxCPM2 Free GPU Engine (Kaggle)", version="2.0.0")

# Enable Full Cross-Origin Access for Lumini Web App
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

VOICE_PROMPTS_DIR = Path(tempfile.gettempdir()) / "lumini_voxcpm_prompts"
VOICE_PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

model = None
model_error = None
sample_rate = 48000

print("⏳ Loading OpenBMB VoxCPM2 Model into GPU...", flush=True)
try:
    from voxcpm import VoxCPM
    model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
    if hasattr(model, 'tts_model') and hasattr(model.tts_model, 'sample_rate'):
        sample_rate = model.tts_model.sample_rate
    print(f"✅ VoxCPM2 Model loaded successfully into GPU! Sample Rate: {sample_rate}Hz", flush=True)
except Exception as e:
    import traceback
    traceback.print_exc()
    model_error = str(e)
    print(f"❌ Model load exception: {e}", flush=True)

@app.get("/")
@app.get("/health")
@app.get("/api/voxcpm/status")
def get_status():
    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
    vram_used = round(torch.cuda.memory_allocated(0) / 1024**3, 2) if cuda_available else 0
    vram_total = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda_available else 0
    
    if model is None:
        return {
            "online": False,
            "ok": False,
            "error": model_error or "VoxCPM2 model is still downloading/initializing",
            "cuda": cuda_available,
            "device": device_name
        }
    return {
        "online": True,
        "ok": True,
        "engine": "VoxCPM2 (OpenBMB 48kHz) — Kaggle Free GPU",
        "cuda": cuda_available,
        "device": f"{device_name} ({vram_used}GB / {vram_total}GB)",
        "sample_rate": sample_rate,
        "supports_zero_shot": True,
        "supports_style_prompting": True
    }

@app.post("/api/voxcpm/clone")
def register_voice_sample(
    name: str = Form(...),
    instruction: Optional[str] = Form("Energetic movie recap narration style"),
    transcript: Optional[str] = Form(None),
    file: UploadFile = File(...)
):
    voice_id = str(uuid.uuid4())
    ext = Path(file.filename or "sample.wav").suffix.lower() or ".wav"
    target_path = VOICE_PROMPTS_DIR / f"{voice_id}{ext}"
    
    content = file.file.read()
    with open(target_path, "wb") as f:
        f.write(content)

    wav_target = VOICE_PROMPTS_DIR / f"{voice_id}.wav"
    if ext != ".wav":
        try:
            os.system(f"ffmpeg -y -i '{target_path}' -ar 48000 -ac 1 '{wav_target}' >/dev/null 2>&1")
        except Exception:
            pass

    return {
        "voiceId": voice_id,
        "name": name,
        "instruction": instruction,
        "message": "Sample registered successfully! Ready for 48kHz Zero-Shot cloning."
    }

def split_burmese_smart(text: str, max_chunk_len: int = 140) -> list[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    raw_sentences = re.split(r'(?<=[။!?\n])\s*', text)
    chunks = []
    current = ""
    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue
        if len(current) + len(s) + 1 <= max_chunk_len:
            current = (current + " " + s).strip() if current else s
        else:
            if current:
                chunks.append(current)
            if len(s) > max_chunk_len:
                parts = re.split(r'(?<=[၊,])\s*', s)
                sub_curr = ""
                for sp in parts:
                    sp = sp.strip()
                    if not sp:
                        continue
                    if len(sub_curr) + len(sp) + 1 <= max_chunk_len:
                        sub_curr = (sub_curr + " " + sp).strip() if sub_curr else sp
                    else:
                        if sub_curr:
                            chunks.append(sub_curr)
                        while len(sp) > max_chunk_len:
                            chunks.append(sp[:max_chunk_len])
                            sp = sp[max_chunk_len:]
                        sub_curr = sp
                current = sub_curr
            else:
                current = s
    if current:
        chunks.append(current)
    return chunks if chunks else [text]

@app.post("/api/voxcpm/synthesize")
def synthesize_speech(
    voice_id: Optional[str] = Form(None),
    text: str = Form(...),
    instruction: Optional[str] = Form(None),
    speed: float = Form(1.0)
):
    if model is None:
        raise HTTPException(status_code=503, detail=f"VoxCPM model not ready: {model_error or 'Initializing'}")
    if not text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")

    audio_path = None
    if voice_id and voice_id != 'default':
        for ext in [".wav", ".mp3", ".webm", ".m4a", ".ogg"]:
            candidate = VOICE_PROMPTS_DIR / f"{voice_id}{ext}"
            if candidate.exists():
                audio_path = str(candidate)
                break

    chunks = split_burmese_smart(text, max_chunk_len=140)
    print(f"🎙️ Synthesizing {len(chunks)} chunks...", flush=True)

    audio_pieces = []
    pause_samples = int(sample_rate * 0.08)
    pause_array = np.zeros(pause_samples, dtype=np.float32)

    try:
        for idx, chunk in enumerate(chunks):
            if not chunk.strip():
                continue
            try:
                if audio_path and os.path.exists(audio_path):
                    wav_chunk = model.generate(
                        text=chunk,
                        reference_wav_path=audio_path,
                        cfg_value=2.0,
                        inference_timesteps=10
                    )
                else:
                    wav_chunk = model.generate(
                        text=chunk,
                        cfg_value=2.0,
                        inference_timesteps=10
                    )
                
                if isinstance(wav_chunk, tuple):
                    wav_chunk = wav_chunk[0]
                if hasattr(wav_chunk, 'cpu'):
                    wav_chunk = wav_chunk.cpu().numpy()
                if isinstance(wav_chunk, np.ndarray):
                    wav_chunk = wav_chunk.flatten().astype(np.float32)
                    audio_pieces.append(wav_chunk)
                    audio_pieces.append(pause_array)
            except Exception as chunk_err:
                print(f"⚠️ Error on chunk {idx}: {chunk_err}", flush=True)
                continue

        if not audio_pieces:
            raise HTTPException(status_code=500, detail="Failed to generate any audio chunks.")

        full_audio = np.concatenate(audio_pieces)
        max_amp = np.max(np.abs(full_audio))
        if max_amp > 1e-5:
            full_audio = full_audio / max_amp * 0.95

        out_io = io.BytesIO()
        sf.write(out_io, full_audio, sample_rate, format='WAV', subtype='PCM_16')
        out_io.seek(0)
        return Response(content=out_io.read(), media_type="audio/wav")
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')
'''

with open('/kaggle/working/app.py', 'w', encoding='utf-8') as f:
    f.write(server_script)

print("✅ FastAPI Engine script written successfully!")

In [ ]:
# ================================================================
# Step 5: Server Run + ngrok Tunnel ချိတ်ဆက်ခြင်း
# ================================================================
import os, subprocess, time, json, urllib.request
from pyngrok import ngrok

# Kill previous runs
!fuser -k 8000/tcp 2>/dev/null; rm -f /tmp/app.log
time.sleep(2)

# 1. Start FastAPI server in background
app_log = open('/tmp/app.log', 'w')
proc_app = subprocess.Popen(
    ['python', '-u', '/kaggle/working/app.py'],
    stdout=app_log, stderr=app_log
)

# 2. Start ngrok tunnel
print("⏳ ngrok Tunnel ချိတ်ဆက်နေပါသည်...", flush=True)
try:
    # Kill existing tunnels
    ngrok.kill()
    time.sleep(1)
    tunnel = ngrok.connect(8000, "http")
    public_url = tunnel.public_url
    # Ensure HTTPS
    if public_url.startswith("http://"):
        public_url = public_url.replace("http://", "https://")
    print(f"✅ ngrok Tunnel: {public_url}")
except Exception as e:
    print(f"❌ ngrok Error: {e}")
    print("👉 ngrok Auth Token ကို စစ်ဆေးပါ")
    public_url = None

# 3. Wait for model to load
print("⏳ Model Weights ကို GPU ပေါ် တင်နေပါသည် (ပထမဆုံးအကြိမ် ~2-3 မိနစ်ခန့် ကြာပါမည်)...", flush=True)
is_ready = False
device_info = "GPU"
for attempt in range(180):  # Up to 3 minutes
    if proc_app.poll() is not None:
        print(f"❌ Server ရပ်တန့်သွားပါသည် (Exit code: {proc_app.returncode})! Log:")
        with open('/tmp/app.log') as f:
            print(f.read())
        break
    try:
        req = urllib.request.urlopen("http://127.0.0.1:8000/api/voxcpm/status", timeout=2)
        if req.status == 200:
            data = json.loads(req.read().decode('utf-8'))
            if data.get('online'):
                is_ready = True
                device_info = data.get('device', 'GPU')
                break
    except Exception:
        pass
    time.sleep(1)

if is_ready and public_url:
    print("\n" + "="*72)
    print("🎉 🟢 VOXCPM2 48KHz FREE GPU SERVER IS 100% ONLINE & READY!")
    print(f"⚡ Device: {device_info}")
    print(f"🌐 Platform: Kaggle (30 hrs/week free GPU)")
    print("="*72)
    print(f"👉 အောက်ပါ URL ကို Copy ယူပြီး Lumini (Voiceover Studio) ထဲတွင် Paste လုပ်ပါ:\n")
    print(f"      {public_url}\n")
    print("="*72 + "\n")
    print("💡 Keep-Alive: Kaggle session ကို 12 နာရီအထိ ဆက်လက်လည်ပတ်စေနိုင်ပါသည်...")
    print("⚠️ ngrok URL ပြောင်းသွားပါက ဒီ Cell ကို ပြန် Run ပါ")
    try:
        while True:
            time.sleep(30)
            if int(time.time()) % 300 < 30:
                print(f"🟢 [Heartbeat {time.strftime('%H:%M:%S')}] Server is healthy & running!")
    except KeyboardInterrupt:
        print("🛑 Server stopped by user.")
        ngrok.kill()
else:
    print("⚠️ Server Initialization Failed. Log:")
    if os.path.exists('/tmp/app.log'):
        with open('/tmp/app.log') as f:
            print(f.read())